# Read MOTEL Ontology (TTL)

This notebook loads the ontology in Python using `rdflib`, then provides:

1. Basic dataset summary
2. Predicate and class inspection
3. Search helpers (keyword / URI fragment)
4. A simple graph visualization of a local neighborhood

In [ ]:
from pathlib import Path

import pandas as pd
from rdflib import Graph, RDF, RDFS, OWL, URIRef, Literal

ONTOLOGY_PATH = Path("motel_ontology.ttl")

g = Graph()
g.parse(ONTOLOGY_PATH, format="ttl")

print(f"Loaded ontology: {ONTOLOGY_PATH}")
print(f"Triples: {len(g):,}")
print(f"Namespaces: {len(list(g.namespaces()))}")


def short_term(term):
    if isinstance(term, URIRef):
        try:
            return g.namespace_manager.normalizeUri(term)
        except Exception:
            return str(term)
    return str(term)

In [ ]:
# Basic summary tables
classes = set(g.subjects(RDF.type, OWL.Class)) | set(g.subjects(RDF.type, RDFS.Class))
object_properties = set(g.subjects(RDF.type, OWL.ObjectProperty))
datatype_properties = set(g.subjects(RDF.type, OWL.DatatypeProperty))
annotation_properties = set(g.subjects(RDF.type, OWL.AnnotationProperty))
individuals = set(g.subjects(RDF.type, OWL.NamedIndividual))

summary_df = pd.DataFrame(
    [
        ["triples", len(g)],
        ["classes", len(classes)],
        ["object_properties", len(object_properties)],
        ["datatype_properties", len(datatype_properties)],
        ["annotation_properties", len(annotation_properties)],
        ["named_individuals", len(individuals)],
    ],
    columns=["metric", "count"],
)

predicate_counts = (
    pd.Series([short_term(p) for _, p, _ in g], name="predicate")
    .value_counts()
    .rename_axis("predicate")
    .reset_index(name="count")
)

print("Ontology summary")
display(summary_df)
print("Top predicates")
display(predicate_counts.head(20))

In [ ]:
def search_ontology(graph: Graph, query: str, max_results: int = 50, case_sensitive: bool = False) -> pd.DataFrame:
    """Search subject/predicate/object text for a query string."""
    q = query if case_sensitive else query.lower()
    rows = []

    for s, p, o in graph:
        s_txt = short_term(s)
        p_txt = short_term(p)
        o_txt = short_term(o)

        hay = " | ".join([s_txt, p_txt, o_txt])
        hay_cmp = hay if case_sensitive else hay.lower()

        if q in hay_cmp:
            rows.append({"subject": s_txt, "predicate": p_txt, "object": o_txt})
            if len(rows) >= max_results:
                break

    return pd.DataFrame(rows)


# Example search
search_df = search_ontology(g, query="technology", max_results=30)
print(f"Matches: {len(search_df)}")
search_df.head(30)

In [ ]:
import math
import plotly.graph_objects as go


def visualize_neighborhood(graph: Graph, center_query: str, max_nodes: int = 40):
    """Visualize a small neighborhood around terms matching center_query."""
    q = center_query.lower()
    centers = []
    for s in set(graph.subjects()):
        s_txt = short_term(s)
        if q in s_txt.lower():
            centers.append(s)
            if len(centers) >= 3:
                break

    if not centers:
        print(f"No node found for query: {center_query}")
        return None

    edges = []
    nodes = set()
    for c in centers:
        nodes.add(c)
        for s, p, o in graph.triples((c, None, None)):
            edges.append((s, p, o))
            nodes.add(s)
            nodes.add(o)
        for s, p, o in graph.triples((None, None, c)):
            edges.append((s, p, o))
            nodes.add(s)
            nodes.add(o)

    # Keep visualization lightweight.
    node_list = list(nodes)[:max_nodes]
    node_set = set(node_list)
    edge_list = [(s, p, o) for s, p, o in edges if s in node_set and o in node_set][: max_nodes * 2]

    if not edge_list:
        print("No edges to visualize for the matched node(s).")
        return None

    labels = [short_term(n) for n in node_list]
    n = len(node_list)
    angles = [2 * math.pi * i / n for i in range(n)]
    pos = {node_list[i]: (math.cos(angles[i]), math.sin(angles[i])) for i in range(n)}

    edge_x = []
    edge_y = []
    for s, _, o in edge_list:
        x0, y0 = pos[s]
        x1, y1 = pos[o]
        edge_x += [x0, x1, None]
        edge_y += [y0, y1, None]

    node_x = [pos[n][0] for n in node_list]
    node_y = [pos[n][1] for n in node_list]

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=edge_x,
            y=edge_y,
            mode="lines",
            line=dict(width=1),
            hoverinfo="none",
            name="edges",
        )
    )
    fig.add_trace(
        go.Scatter(
            x=node_x,
            y=node_y,
            mode="markers+text",
            text=labels,
            textposition="top center",
            marker=dict(size=10),
            name="nodes",
            hovertext=labels,
            hoverinfo="text",
        )
    )
    fig.update_layout(
        title=f"Ontology neighborhood for '{center_query}'",
        showlegend=False,
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        height=700,
    )
    return fig


fig = visualize_neighborhood(g, center_query="Technology", max_nodes=35)
if fig is not None:
    fig.show()